# FABGal - Fluorescence Analysis of senescence-associated B-Gal 

This Jupyter Notebook is designed to guide the user through the analysis of fluorescence images of cell culture or tissue senescence-associated beta-galactosidase (SA-βgal) assays, enabling a quantitative approach to this routine technique in aging studies.

The FABGal workflow first loads user images located in the specified **input_folder**. Input images must be in TIFF format (.tif or .tiff), and include information about physical pixel sizes. 


## Parameters



In [17]:
delete_intermediate_files = True # True if intermediate files generated by BiaPy are to be removed at the end of the execution.
substract_background = True # True if the 

In [ ]:

# Files
input_folder="./individuals/" # Input images directory (Must exist)
experiment_name = "Experiment1" # Experiment name for naming output files generated in the analysis.

# Image parameters
nuclei_ch = 0 # Channel containing nuclei stain signal. Channels follow a 0-based numbering (first channel is 0).
bgal_ch = 1 # Channel containing BGal stain signal.

# Bgal parameters
bgal_th = 45 # Minimum value of BGal stain signal to be considered positive. Default: 45.

# Substract background parameters
sbg_rad = 31 # Radius of substract background. Should be adjusted depending on nuclei size. Default: 31.

# Biapy parameters
config_path = "./instance_segmentation.yaml"            # Path to BiaPy YAML configuration file
result_dir = "./biapy_output"                 # Directory to store BiaPy results
job_name = experiment_name                      # Name of BiaPy job. Dafaults to experiment name.
run_id = 1                                      # Run ID for logging/versioning
gpu = "0"                                       # GPU to use (as string, e.g., "0"). Obtained from nvidia-smi command.

## Measure B-Gal and create tiffs for biapy


In [19]:
from skimage.restoration import rolling_ball
from skimage.transform import rescale
from skimage.util import img_as_ubyte, img_as_float
from scipy.ndimage import uniform_filter
import numpy as np

from numpy.typing import NDArray
from typing import List

def calculate_bgal(img, bgal_ch: int, bgal_thmin: int, bgal_thmax: int = 254) -> List[float]:
    """
    Calculates the area and intensity of B-gal positive regions in an image.

    This function masks pixels falling within the specific threshold range 
    and calculates the total pixel count, physical area, and intensity sum.

    Args:
        img: A BioImage object containing pixel size metadata.
        bgal_ch (int): The channel index for the B-gal signal.
        bgal_thmin (int): The minimum pixel intensity threshold.
        bgal_thmax (int, optional): The maximum pixel intensity threshold. 
            Defaults to 254 to exclude burned (saturated) pixels at 255 
            where real values are unknown.

    Returns:
        list: A list containing three values:
            1. Number of positive pixels (NpxPos)
            2. Physical area of positive region (NpxPos * pixel_area)
            3. Total intensity of positive pixels (Intens)
    """
    # Calculate area of a single pixel (Y size * X size)
    pxarea = img.physical_pixel_sizes[1] * img.physical_pixel_sizes[2]
    
    # Extract image data for the specific channel
    imgdata = img.get_image_data("YX", C=bgal_ch)
    
    # Create a boolean mask for pixels within the threshold
    mask = (imgdata >= bgal_thmin) & (imgdata <= bgal_thmax)
    
    # Calculate stats
    NpxPos = np.sum(mask).item()
    Intens = np.sum(imgdata[mask == 1]).item()
    
    return [NpxPos, NpxPos * pxarea, Intens]

## Function to substract background
def subtract_background(img, radius : int, scale : int = 8) -> NDArray[np.uint8]:
    """
    Subtracts the background signal of an input image using the rolling ball algorithm.

    This function smooths the image using an uniform filter, estimates background signal via a downscaled 
    rolling ball, and returns the background-subtracted image as a numpy array of uint8 format in order to be saved.

    Args:
        img (np.ndarray): The input image array (pixel data).
        radius (int): Radius to be applied in the rolling_ball algorithm.
        scale (int, optional): Factor by which the image is downscaled to speed up 
            background estimation. Defaults to 8.

    Returns:
        np.ndarray: The background-subtracted image as ubyte (uint8).

    """
    img = uniform_filter(img, size=3)
    img = img_as_float(img)
    bg = rescale(img, 1/scale, order=1)
    bg = rolling_ball(bg, radius=radius, nansafe=False)
    bg = rescale(bg, scale, order=1)
    return img_as_ubyte(img-bg)

In [20]:
from bioio import BioImage
from bioio_base.exceptions import UnsupportedFileFormatError
import os
import sys
from bioio_ome_tiff.writers import OmeTiffWriter

# Check if biapy_input folder exists, if not create it
if not os.path.isdir("./biapy_input"):
    os.makedirs("./biapy_input")

# Read all files
try:
    myfiles = os.listdir(input_folder)
except FileNotFoundError as e:
    raise FileNotFoundError("ERROR: Input directory does not exist. Please check input_folder variable.") from e

myfiles = [f for f in myfiles if f.lower().endswith(('.tif', '.tiff'))]


# Iterate over all files
bres = {}
for inf in myfiles:
    print("Doing file " + inf)

    try:
        img = BioImage(os.path.join(input_folder, inf))
    except UnsupportedFileFormatError as e:
        raise UnsupportedFileFormatError("\nERROR: Image is not in TIFF format.") from e
    except Exception as e:
        raise Exception(f"\nERROR: Could not open {inf}: {e}") from e
    
    if img.physical_pixel_sizes.X is None:
        raise Exception("\nERROR: Image does not have pixel physical sizes.")


    print(f"Image pixel sizes are {img.physical_pixel_sizes}")

    # Measure bgal area
    bres[inf] = calculate_bgal(img,bgal_ch,bgal_th)

    # Save tiff for biapy input
    if(substract_background):
        OmeTiffWriter.save(subtract_background(img.get_image_data("YX",C=nuclei_ch), radius=sbg_rad), 
                    "./biapy_input/" + inf, dim_order="YX")
    else:
       OmeTiffWriter.save(img.get_image_data("YX",C=nuclei_ch), 
                    "./biapy_input/" + inf, dim_order="YX")


# Save bgal results
bgal_f = open(experiment_name + "_BGal_results.tsv",'w')
bgal_f.write("File\tNpxPos\tAreaPos\tBgal_intensity\n")
for k, (npx, area, intens) in bres.items():
    bgal_f.write(f"{k}\t{npx}\t{area}\t{intens}\n")
# for k,v in bres.items():
    # bgal_f.write(str(k) + "\t" + "\t".join([str(i) for i in v]))
bgal_f.close()

Attempted file (/home/droiva/Documents/Projects/SAbGal_biapy/individuals/20250905_Plate_seeded_20250902_BGal-02-Scene-12-P4-E06.tif) load with reader: <class 'bioio_ome_tiff.reader.Reader'> failed with error: bioio-ome-tiff does not support the image: '/home/droiva/Documents/Projects/SAbGal_biapy/individuals/20250905_Plate_seeded_20250902_BGal-02-Scene-12-P4-E06.tif'. Failed to parse XML for the provided file. Error: not well-formed (invalid token): line 1, column 6


[15:28:15.601217] Doing file 20250905_Plate_seeded_20250902_BGal-02-Scene-12-P4-E06.tif
[15:28:15.609012] Image pixel sizes are PhysicalPixelSizes(Z=None, Y=0.5476193344672704, X=0.5476193344672704)


## Run Biapy

In [21]:
# Create and run the BiaPy job

from biapy import BiaPy
biapy = BiaPy(config_path, result_dir=result_dir, name=job_name, run_id=run_id, gpu=gpu)
biapy.run_job()

[15:28:32.805545] Date     : 2026-01-21 15:28:32
[15:28:32.805589] Arguments: Namespace(config='./instance_segmentation.yaml', result_dir='./biapy_output', name='Experiment1', run_id=1, gpu='0', world_size=1, local_rank=-1, dist_on_itp=False, dist_url='env://', dist_backend='nccl')
[15:28:32.805596] Job      : Experiment1_1
[15:28:32.805602] BiaPy    : 3.6.8
[15:28:32.805608] Python   : 3.10.19 | packaged by conda-forge | (main, Oct 22 2025, 22:29:10) [GCC 14.3.0]
[15:28:32.805615] PyTorch  : 2.9.1+cu128
[15:28:32.810117] The following changes were made in order to adapt the input configuration:
[15:28:32.811995] Not using distributed mode
[15:28:32.812983] WARNING: seems that the channels requested are custom so BiaPy did not fill some varibles by default.
You will need to fill the following variables:
    - PROBLEM.INSTANCE_SEG.WATERSHED.SEED_CHANNELS
    - PROBLEM.INSTANCE_SEG.WATERSHED.SEED_CHANNELS_THRESH
    - PROBLEM.INSTANCE_SEG.WATERSHED.TOPOGRAPHIC_SURFACE_CHANNEL
    - PROBL

[15:28:49.445810] ### MERGE-OV-CROP ###
[15:28:49.445831] Merging (143, 256, 256, 2) images into (1, 2056, 2464, 2) with overlapping . . .
[15:28:49.445838] Minimum overlap selected: (0, 0)
[15:28:49.445848] Padding: (32, 32)
[15:28:49.447044] Real overlapping (%): (0.010416666666666666, 0.026041666666666668)
[15:28:49.447058] Real overlapping (pixels): (2.0, 5.0)
[15:28:49.447064] (13, 11) patches per (x,y) axis
[15:28:49.501565] **** New data shape is: (1, 2056, 2464, 2)
[15:28:49.501586] ### END MERGE-OV-CROP ###
[15:28:49.501740] ### MERGE-OV-CROP ###
[15:28:49.501748] Merging (143, 256, 256, 1) images into (1, 2056, 2464, 1) with overlapping . . .
[15:28:49.501755] Minimum overlap selected: (0, 0)
[15:28:49.501760] Padding: (32, 32)
[15:28:49.503889] Real overlapping (%): (0.010416666666666666, 0.026041666666666668)
[15:28:49.503903] Real overlapping (pixels): (2.0, 5.0)
[15:28:49.503909] (13, 11) patches per (x,y) axis
[15:28:49.521721] **** New data shape is: (1, 2056, 2464, 1)


[15:28:49.595506] Creating instances with watershed . . .
[15:28:49.699874] Thresholds used: {'seed': [0.7, 0.6], 'growth_mask': [0.23729699850082397]}
[15:28:49.873001] Saving (1, 1, 2056, 2464, 1) data as .tif in folder: ./biapy_output/Experiment1/results/Experiment1_1/per_image_instances


[15:28:49.880458] Checking the properties of instances . . .


[15:28:50.064690] Removed 0 instances by properties ([]), 631 instances left
[15:28:50.069576] Releasing memory . . .
[15:28:50.069600] #############
[15:28:50.069606] #  RESULTS  #
[15:28:50.069612] #############
[15:28:50.069617] The values below represent the averages across all test samples
[15:28:50.069633] Instance segmentation specific metrics:
[15:28:50.069656] FINISHED JOB Experiment1_1 !!


## Process BiaPy's output

In [22]:
import pandas as pd
from skimage import io
from skimage.color import label2rgb
import numpy as np
import os
import shutil


# Biapy's output folder
mydir=os.path.join(result_dir,job_name,"results",job_name + "_" + str(run_id),"per_image_instances")

# Nuclei stats output file
biapystatsfile = experiment_name + "_BiaPy_results.csv"

### Process BiaPy output images

Read all label images, convert then to RGB, save as png and delete original tif

In [23]:
myfiles = os.listdir(mydir)
myfiles = [f for f in myfiles if f.endswith(".tif") ]

for inf in myfiles:
    img = io.imread(os.path.join(mydir,inf))
    img = label2rgb(img)*255
    img = img.astype(np.uint8)
    io.imsave(os.path.join(mydir,inf.replace('.tif','.png')),img)
    os.remove(os.path.join(mydir,inf))

### Process nuclei stats

Read all CSVs, add file name, concatenate, save and delete original

In [24]:
myfiles = os.listdir(mydir)
myfiles = [f for f in myfiles if f.endswith("full_stats.csv") ]

#list of csv with file name column
dfs = []
for inf in myfiles:
    df = pd.read_csv(os.path.join(mydir,inf))
    df["File"] = inf
    dfs.append(df)

#concatenate
result = pd.concat(dfs, ignore_index=True)
result = result.drop(columns=["conditions"])
result.to_csv(biapystatsfile)

# Delete original
for inf in myfiles:
    os.remove(os.path.join(mydir,inf))

### Delete extra files generated by Biapy (optional)

Delete intermediate files and files not used in this analysis

In [25]:
if delete_intermediate_files:
    interf = os.path.join(result_dir,job_name,"results",job_name + "_" + str(run_id),"per_image")
    extraf = os.path.join(result_dir,job_name,"results",job_name + "_" + str(run_id),"per_image_post_processing")

    shutil.rmtree(interf)
    shutil.rmtree(extraf)